### GOALS/Infra needed

1. S3 Buckets with boto3.
2. IAM Roles and users
3. Complete infra of AWS Sagemaker - Training and endpoints


In [1]:
import sagemaker
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
import os

load_dotenv()
import boto3
import pandas as pd
region = 'us-east-2'
boto = boto3.client('sagemaker',region_name=region)
boto_session = boto3.Session(
                    region_name=region,
                    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
                )
session = sagemaker.Session(boto_session=boto_session)


sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\theVa\AppData\Local\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\theVa\AppData\Local\sagemaker\sagemaker\config.yaml


In [2]:
print(session.boto_region_name)  # should print us-east-2
print(session._region_name)


us-east-2
us-east-2


In [3]:
print(region)
file_path = 'mob_price_classification_train.csv'
df = pd.read_csv(file_path)



us-east-2


In [4]:
df.head()
df.shape

(2000, 21)

In [5]:
df.isnull().sum()


battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [6]:
df['price_range'].value_counts()
cols = list(df.columns)
features = cols[:-1]
label = cols[-1]


In [7]:
X = df[features]
y = df[label]


In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y)

train_df = pd.DataFrame(X_train)
test_df = pd.DataFrame(X_test)


train_df= pd.concat([X_train,y_train], axis=1)
test_df = pd.concat([X_test,y_test],axis=1)
train_df.to_csv('train_V1.csv',index=False)
test_df.to_csv('test_V1.csv',index=False)

print(f'X_train shape={X_train.shape}')
print(f'Y_train shape={y_train.shape}')
print(f'X_test shape={X_test.shape}')
print(f'y_test shape={y_test.shape}')

X_train shape=(1500, 20)
Y_train shape=(1500,)
X_test shape=(500, 20)
y_test shape=(500,)


In [9]:
pd.read_csv('test_V1.csv').shape  # should be (500, 21)


(500, 21)

In [10]:
pd.read_csv('train_V1.csv').shape  # should be (1500, 21)


(1500, 21)

In [11]:
# Run this to verify the saved files are clean
import pandas as pd
print(pd.read_csv('train_V1.csv').shape)  # must be (1500, 21)
print(pd.read_csv('test_V1.csv').shape)   # must be (500, 21)
print(pd.read_csv('train_V1.csv').columns.tolist())  # last column must be 'price_range'


(1500, 21)
(500, 21)
['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g', 'touch_screen', 'wifi', 'price_range']


In [56]:
bucket_name = 'hardyssagemakerdemo'
#send data to s3, sagemaker will take data from s3

sk_prefix='sagemaker/mobile_price_classification/sklearncontainer'

train_path = session.upload_data(path = 'train_V1.csv', bucket = bucket_name, key_prefix = sk_prefix+'/train')
test_path = session.upload_data(path='test_V1.csv',bucket=bucket_name,key_prefix=sk_prefix+'/test')

print(train_path)

s3://hardyssagemakerdemo/sagemaker/mobile_price_classification/sklearncontainer/train/train_V1.csv


### here's the script used by sagemaker used to train 

In [57]:
%%writefile script.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix,precision_score

import sklearn
import joblib

import boto3
from io import StringIO

import argparse
import os
import numpy as np
import pandas as pd

def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir,'model.joblib'))
    return clf

if __name__=='__main__':
    print('Info extracting args')
    parser = argparse.ArgumentParser()
    ##hyperparameter for random forest

    parser.add_argument('--n_estimators',type=int, default=100)
    parser.add_argument('--random_state',type=int, default=0)


    #output dirs declaration
    parser.add_argument('--model-dir',type=str,default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train',type=str,default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--test',type=str,default=os.environ.get('SM_CHANNEL_TEST'))
    parser.add_argument('--train-file',type=str,default='train_V1.csv')
    parser.add_argument('--test-file',type=str,default='test_V1.csv')


    args, _ = parser.parse_known_args()
    print(f'sklearn version={sklearn.__version__}')
    print(f'joblib version={joblib.__version__}')

    print(f'[INFO] reading data')
    print()

    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))


    # train_df = train_df.fillna(train_df.median())
    # test_df  = test_df.fillna(train_df.median())  # use train median for both
    cols = list(train_df.columns)
    features = cols[:-1]
    label = cols[-1]

    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]
    

    print(f'X_train shape={X_train.shape}')
    print(f'Y_train shape={y_train.shape}')
    print(f'X_test shape={X_test.shape}')
    print(f'y_test shape={y_test.shape}')
    model = RandomForestClassifier(
                n_estimators = args.n_estimators,
                random_state=args.random_state,
                verbose=2,
                n_jobs=1
            )
    model.fit(X_train,y_train)


    model_path = os.path.join(args.model_dir,'model.joblib')

    joblib.dump(model,model_path)

    print(f'Saved model at path={model_path}')
    y_pred_test = model.predict(X_test)

    test_acc = accuracy_score(y_test,y_pred_test)
    test_rep = classification_report(y_test,y_pred_test)


    print()
    print('---METRICS RESULTS for TESTING DATA---')
    print(f'Total rows are:{ X_test.shape[0]}')
    print(f'[TESTING] Model Accuracy is: {test_acc}')
    print(f'[TESTING] Testing report: {test_rep}')


Overwriting script.py


In [3]:
#AWS Sagemaker entry point to execute training script
from sagemaker.sklearn.estimator import SKLearn

FRAMEWORK_VERSION ='0.23-1'
sklearn_estimator = SKLearn(
    entry_point='script.py',
    role='arn:aws:iam::927198448836:role/sagemakeraccessrole',
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version=FRAMEWORK_VERSION,
    base_job_name='RF-custom-sklearn',
    hyperparameters = {
        'n_estimators':100,
        'random_state':0
    },
    use_spot_instances=True,
    max_wait=7200,
    max_run=3600,
    sagemaker_session=session
)

In [59]:

from sagemaker.inputs import TrainingInput

train_input = TrainingInput('s3://hardyssagemakerdemo/sagemaker/mobile_price_classification/sklearncontainer/train', content_type='text/csv')
test_input  = TrainingInput('s3://hardyssagemakerdemo/sagemaker/mobile_price_classification/sklearncontainer/test',  content_type='text/csv')

sklearn_estimator.fit({'train': train_input, 'test': test_input})

INFO:sagemaker:Creating training-job with name: RF-custom-sklearn-2026-04-25-21-41-55-654


Using provided s3_resource
2026-04-25 21:41:56 Starting - Starting the training job...
2026-04-25 21:42:11 Starting - Preparing the instances for training...
2026-04-25 21:42:57 Downloading - Downloading the training image......
2026-04-25 21:43:54 Training - Training image download completed. Training in progress.
2026-04-25 21:43:54 Uploading - Uploading generated training model2026-04-25 21:43:47,041 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-04-25 21:43:47,045 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-25 21:43:47,084 sagemaker_sklearn_container.training INFO     Invoking user training script.
2026-04-25 21:43:47,278 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-25 21:43:47,290 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-04-25 21:43:47,301 sagemaker-training-toolkit INFO     No GPUs detected (normal 

In [60]:
sklearn_estimator.latest_training_job.wait(logs='None')

artifact = boto.describe_training_job(TrainingJobName=sklearn_estimator.latest_training_job.name)['ModelArtifacts']['S3ModelArtifacts']


2026-04-25 21:44:06 Starting - Preparing the instances for training
2026-04-25 21:44:06 Downloading - Downloading the training image
2026-04-25 21:44:06 Training - Training image download completed. Training in progress.
2026-04-25 21:44:06 Uploading - Uploading generated training model
2026-04-25 21:44:06 Completed - Training job completed


In [61]:
artifact

's3://sagemaker-us-east-2-927198448836/RF-custom-sklearn-2026-04-25-21-41-55-654/output/model.tar.gz'

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

model_name='Custom-sklearn-model'+strftime('%Y-%m-%d-%H-%M-%S',gmtime())
model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role='arn:aws:iam::927198448836:role/sagemakeraccessrole',
    entry_point='script.py',
    framework_version=FRAMEWORK_VERSION,
    sagemaker_session=session
)

In [13]:
#If you wanna pull the model with a hardcoded artifact
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

artifact = 's3://sagemaker-us-east-2-927198448836/RF-custom-sklearn-2026-04-25-21-41-55-654/output/model.tar.gz'

model_name = 'Custom-sklearn-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
FRAMEWORK_VERSION ='0.23-1'
model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role='arn:aws:iam::927198448836:role/sagemakeraccessrole',
    entry_point='script.py',
    framework_version=FRAMEWORK_VERSION,
    sagemaker_session=session
)


In [15]:
model


In [ ]:
endpoint_name='Custom-sklearn-model-'+strftime('%Y-%m-%d-%H-%M-%S',gmtime())

print('Endpoint={}'.format(endpoint_name))

predictor = model.deploy(
    initial_instance_count=1,
    instance_type='ml.m4.xlarge',
    endpoint_name=endpoint_name
)

Endpoint=
-----!

In [14]:
X_test[0:2]

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,pc,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi
112,867,0,1.4,1,0,1,4,0.7,135,6,1,70,1974,790,13,6,3,1,1,0
1898,684,1,0.9,1,3,1,63,1.0,157,5,9,159,1738,3756,17,5,12,1,1,1


In [ ]:
predictor.predict(X_test[0:2].values.tolist())

array([0, 3], dtype=int64)

In [25]:
session.delete_endpoint(endpoint_name=endpoint_name)